New mosaics (and new bands), and updated data version numbers

In [1]:
import importlib.metadata
import re
import shutil
import time
from pathlib import Path

import ee
import gdown
import pandas as pd
import polars as pl

In [2]:
ee.Initialize(project='mapbiomas-india')

Configuration
Region IDs, sample versions, asset folder and local data paths live in `config.py`, shared with the other notebooks. Update versions there only, then restart the kernel.

In [3]:
from config import (
    regions,
    region_id_map,
    samples_version_map,
    samples_folder,
    territory_name,
    DATA_DIR,
    AGGREGATED_DIR,
    region_data_path,
)

1. Get total samples per year for each region

In [4]:
def get_region_sample_counts(
    year: int,
    region_id_map: dict,
    samples_version_map: dict,
    samples_folder: str,
    territory_name: str
) -> pd.DataFrame:
    """
    Computes total sample counts per region for a specified year
    and returns a formatted DataFrame.
    """
    records = []

    for region, version in samples_version_map.items():
        region_id = region_id_map.get(region)
        if region_id is None:
            raise ValueError(f"Region '{region}' is in samples_version_map but not in region_id_map.")

        asset_name = f"{territory_name}_training_samples_region{region_id}_v{version}_merged"
        asset_path = f"{samples_folder.rstrip('/')}/{asset_name}"

        try:
            fc = ee.FeatureCollection(asset_path).filter(ee.Filter.eq('year', year))
            count = fc.size().getInfo()
        except ee.EEException as e:
            print(f"WARNING: could not count samples for {region} v{version} ({asset_path}): {e}")
            count = None

        if count == 0:
            print(f"WARNING: 0 samples for {region} v{version} in {year}. "
                  f"Check the year exists in the asset and that 'year' is stored as a number.")

        records.append({
            'Region': region,
            'Region ID': region_id,
            'Version': version,
            'Year': year,
            'Total Samples': count
        })

    df = pd.DataFrame(records)
    return df

In [5]:
get_region_sample_counts(
    year = 2025,
    region_id_map = region_id_map,
    samples_version_map = samples_version_map,
    samples_folder = samples_folder,
    territory_name = territory_name
)

,Region,Region ID,Version,Year,Total Samples
0,HIM,1,7,2025,49007
1,IGP,2,7,2025,82587
2,DES,3,8,2025,117387
3,PEN,4,6,2025,58316
4,NEI,5,7,2025,157807
5,WGR,6,5,2025,63193
6,ICR,7,7,2025,94052


In [6]:
get_region_sample_counts(
    year = 2015,
    region_id_map = region_id_map,
    samples_version_map = samples_version_map,
    samples_folder = samples_folder,
    territory_name = territory_name
)

,Region,Region ID,Version,Year,Total Samples
0,HIM,1,7,2015,48657
1,IGP,2,7,2015,82587
2,DES,3,8,2015,117387
3,PEN,4,6,2015,58316
4,NEI,5,7,2015,133443
5,WGR,6,5,2015,63193
6,ICR,7,7,2015,94052


2. Export samples from GEE to Drive

In [7]:
def export_samples_to_drive(
    regions: list,
    region_id_map: dict,
    samples_version_map: dict,
    samples_folder: str,
    territory_name: str,
    years: list = None,
    drive_folder: str = "GEE_EXPORTS"
):
    """
    Exports training samples to Drive and returns the started tasks.
    If `years` is None or empty, exports the complete un-filtered dataset (all years).
    """
    tasks = []
    for region in regions:
        if region not in samples_version_map:
            print(f"Skipping '{region}': no entry in samples_version_map")
            continue
        if region not in region_id_map:
            raise ValueError(f"Region '{region}' is not in region_id_map.")
        region_id = region_id_map[region]
        version = samples_version_map[region]
        asset_path = f"{samples_folder.rstrip('/')}/{territory_name}_training_samples_region{region_id}_v{version}_merged"

        # Case 1: Export complete asset across all years
        if not years:
            fc = (ee.FeatureCollection(asset_path)
                  .select(propertySelectors=['.*'], retainGeometry=False))
            description = f"{region}_all_years_v{version}"
            task = ee.batch.Export.table.toDrive(
                collection=fc,
                description=description,
                folder=drive_folder,
                fileNamePrefix=description,
                fileFormat='CSV'
            )
            task.start()
            print(f"Complete asset export task queued: {description} (Task ID: {task.id})")
            tasks.append(task)
            continue

        # Case 2: Export specific filtered years
        for year in years:
            fc = (ee.FeatureCollection(asset_path)
                  .filter(ee.Filter.eq('year', year))
                  .select(propertySelectors=['.*'], retainGeometry=False))

            description = f"{region}_{year}_v{version}"
            task = ee.batch.Export.table.toDrive(
                collection=fc,
                description=description,
                folder=drive_folder,
                fileNamePrefix=description,
                fileFormat='CSV'
            )
            task.start()
            print(f"Yearly export task queued: {description} (Task ID: {task.id})")
            tasks.append(task)

    return tasks

In [ ]:
# target_regions = ['DES']
# target_years = [1995,2005]

# export_tasks = export_samples_to_drive(
#     regions=target_regions,
#     years=target_years,
#     drive_folder="GEE_EXPORTS/IOLN_new_mosaics",
#     region_id_map=region_id_map,
#     samples_version_map=samples_version_map,
#     samples_folder=samples_folder,
#     territory_name=territory_name,
# )

Export all years for the selected regions

In [ ]:
target_regions = ['NEI','PEN']
target_years = None

In [ ]:
export_tasks = export_samples_to_drive(
    regions=target_regions,
    years=None,
    drive_folder="GEE_EXPORTS/IOLN_new_mosaics_complete",
    region_id_map=region_id_map,
    samples_version_map=samples_version_map,
    samples_folder=samples_folder,
    territory_name=territory_name,
)

Wait for the export tasks to finish
Polls GEE until every task has completed, failed or been cancelled. Stops with an error listing any failed tasks.

In [ ]:
def wait_for_export_tasks(tasks: list, poll_seconds: int = 60):
    """Wait for GEE export tasks to finish; raise if any of them failed or were cancelled."""
    if not tasks:
        print("No export tasks to wait for.")
        return

    pending = {task.id: task for task in tasks}
    failed = []

    while pending:
        for task_id, task in list(pending.items()):
            status = task.status()
            state = status.get('state')
            description = status.get('description', task_id)

            if state == 'COMPLETED':
                print(f"Completed: {description}")
                del pending[task_id]
            elif state in ('FAILED', 'CANCELLED', 'CANCEL_REQUESTED'):
                print(f"{state}: {description}: {status.get('error_message', 'no error message')}")
                failed.append(description)
                del pending[task_id]

        if pending:
            print(f"{len(pending)} task(s) still running, checking again in {poll_seconds}s...")
            time.sleep(poll_seconds)

    if failed:
        raise RuntimeError(f"{len(failed)} export task(s) did not complete: {failed}")
    print("All export tasks completed.")

In [ ]:
wait_for_export_tasks(export_tasks)

3. Download from Drive to local
Downloads the Drive export folder into a staging folder, then moves the `<REGION>_all_years_v<VERSION>.csv` files into the input folder used for partitioning. Drive duplicates such as `DES_all_years_v8 (1).csv` and unexpected files are reported and not moved.

In [7]:
# Google Drive folder that the exports above write to (from the folder's share link)
DRIVE_FOLDER_ID = '1eM0jk5BsoaESATb_f9fsBT7N4mhj2ykF'
RUN_DOWNLOAD = False  # set to True to download

EXPORT_FILENAME_PATTERN = re.compile(r"^[A-Za-z0-9]+_all_years_v\d+\.csv$")
GDOWN_FOLDER_FILE_LIMIT = 50  # gdown.download_folder only fetches up to 50 files per folder


def download_drive_exports(folder_id: str, input_folder: Path):
    """Download a Drive folder via gdown and move the export CSVs into input_folder."""
    staging_dir = input_folder.parent / "_download_staging"
    if staging_dir.exists():
        print(f"Removing leftover staging folder from an earlier run: {staging_dir}")
        shutil.rmtree(staging_dir)
    staging_dir.mkdir(parents=True)
    input_folder.mkdir(parents=True, exist_ok=True)

    downloaded = gdown.download_folder(id=folder_id, output=str(staging_dir), quiet=False, use_cookies=False)
    if not downloaded:
        raise RuntimeError(f"gdown downloaded nothing from Drive folder {folder_id}. Check the ID and sharing settings.")
    if len(downloaded) >= GDOWN_FOLDER_FILE_LIMIT:
        print(f"WARNING: {len(downloaded)} files downloaded, which is gdown's per-folder limit. Some files may be missing.")

    moved, not_moved = [], []
    for path in map(Path, downloaded):
        if not path.is_file():
            continue
        if not EXPORT_FILENAME_PATTERN.match(path.name):
            not_moved.append(f"{path.name} (unexpected name, possibly a Drive duplicate)")
            continue
        destination = input_folder / path.name
        if destination.exists():
            not_moved.append(f"{path.name} (already in {input_folder})")
            continue
        path.rename(destination)
        moved.append(path.name)

    shutil.rmtree(staging_dir)
    print(f"Moved into {input_folder}: {moved}")
    if not_moved:
        print(f"WARNING: not moved: {not_moved}")


if RUN_DOWNLOAD:
    download_drive_exports(DRIVE_FOLDER_ID, AGGREGATED_DIR)

4. Convert all-years regional CSV or Parquet to year-partitioned Parquet
Partitions each regional dataset by `year` into compact, zstd-compressed Parquet files.

Layout: `data/regionwise_yearwise_samples/<REGION>/v<VERSION>/year=YYYY/*.parquet`

The version is read from the source filename (e.g. `DES_all_years_v8.csv` → `v8`) and must match `samples_version_map`.

Each region is written to a temporary `_tmp_v<VERSION>` folder and only renamed to `v<VERSION>` after these checks pass: per-year row counts (source vs written), column names and types, and per-year counts against the GEE asset (`VERIFY_AGAINST_GEE`). The source file is deleted only after that.

In [10]:
input_folder = AGGREGATED_DIR
output_base_dir = DATA_DIR
output_base_dir.mkdir(parents=True, exist_ok=True)

# Compare per-year row counts against the GEE asset before deleting the source file
VERIFY_AGAINST_GEE = True

# GEE export artefacts that carry no sample data
GEE_EXTRA_COLUMNS = ["system:index", ".geo"]


def gee_rows_per_year(region: str, version: str) -> dict:
    """Per-year sample counts in the GEE asset, in a single getInfo call."""
    region_id = region_id_map[region]
    asset_path = f"{samples_folder.rstrip('/')}/{territory_name}_training_samples_region{region_id}_v{version}_merged"
    histogram = ee.FeatureCollection(asset_path).aggregate_histogram('year').getInfo()
    return {int(float(year)): int(count) for year, count in histogram.items()}


def partitioned_rows_per_year(partition_dir: Path) -> dict:
    """Per-year row counts read from the year=YYYY folder names."""
    counts = {}
    for year_dir in partition_dir.glob("year=*"):
        year = int(year_dir.name.split("=", 1)[1])
        counts[year] = (
            pl.scan_parquet(year_dir / "*.parquet", hive_partitioning=False)
            .select(pl.len()).collect().item()
        )
    return counts


def partitioned_schema(partition_dir: Path) -> dict:
    """Column names and types of the partitioned files (the year column is checked separately)."""
    first_file = next(partition_dir.glob("year=*/*.parquet"), None)
    if first_file is None:
        raise FileNotFoundError(f"No Parquet files were written under {partition_dir}.")
    schema = pl.read_parquet_schema(first_file)
    return {name: dtype for name, dtype in schema.items() if name != "year"}


# Remove temporary folders left behind by an interrupted run
for leftover in output_base_dir.glob("*/_tmp_v*"):
    print(f"Removing leftover temporary folder from an interrupted run: {leftover}")
    shutil.rmtree(leftover)

# Discover both Parquet and CSV files
supported_extensions = ("*.parquet", "*.csv")
data_files = []
for ext in supported_extensions:
    data_files.extend(input_folder.glob(ext))
data_files = sorted(data_files)

print(f"Discovered {len(data_files)} source file(s): {[f.name for f in data_files]}")

for filepath in data_files:
    # Avoid processing a file if it was deleted as a counterpart in a previous iteration
    if not filepath.exists():
        print(f"Skipping {filepath.name}: already removed as the counterpart of a processed file.")
        continue

    # Extract region code (e.g., 'DES' from 'DES_all_years_v8.parquet')
    match = re.match(r"^([A-Za-z0-9]+)", filepath.stem)
    if not match:
        print(f"Skipping file with unrecognized naming format: {filepath.name}")
        continue

    region = match.group(1)

    # Version comes from the filename (e.g. 'DES_all_years_v8') and must match samples_version_map
    version_match = re.search(r"_v(\d+)", filepath.stem)
    if not version_match:
        print(f"Skipping {filepath.name}: no '_v<number>' version in filename.")
        continue
    file_version = version_match.group(1)

    expected_version = samples_version_map.get(region)
    if expected_version is None:
        print(f"Skipping {filepath.name}: region '{region}' not in samples_version_map.")
        continue
    if file_version != expected_version:
        print(f"Skipping {filepath.name}: file is v{file_version}, samples_version_map expects v{expected_version}.")
        continue

    region_partition_dir = region_data_path(region, file_version)
    tmp_partition_dir = output_base_dir / region / f"_tmp_v{file_version}"

    # Skip if partitioned data already exists for this region and version.
    # Only complete, verified data ever reaches this folder (see the rename below).
    if region_partition_dir.exists() and any(region_partition_dir.glob("year=*")):
        print(f"Partitioned data for '{region}' v{file_version} already exists at {region_partition_dir}. Skipping.")
        continue

    print(f"\nProcessing {filepath.name} for region '{region}' (v{file_version})...")

    # Load data via lazy scan. CSV types are inferred from the whole file, not the first rows.
    if filepath.suffix.lower() == ".parquet":
        lf = pl.scan_parquet(filepath)
    elif filepath.suffix.lower() == ".csv":
        lf = pl.scan_csv(filepath, infer_schema_length=None)
    else:
        print(f"Skipping {filepath.name}: unsupported file type '{filepath.suffix}'.")
        continue

    schema = lf.collect_schema()
    if "year" not in schema:
        raise KeyError(f"'year' column missing in {filepath.name}")

    # Drop GEE export artefacts; cast year to int32 for clean folder names (year=1995/ vs year=1995.0/)
    lf = (
        lf.drop([c for c in GEE_EXTRA_COLUMNS if c in schema])
          .with_columns(pl.col("year").cast(pl.Int32))
    )

    df = lf.collect()
    raw_row_count = df.height
    source_rows_per_year = {
        int(year): int(n) for year, n in df.group_by("year").len().iter_rows()
    }
    source_schema = {name: dtype for name, dtype in df.schema.items() if name != "year"}
    print(f"  Found {raw_row_count:,} rows across {len(source_rows_per_year)} years.")

    # Write Hive-partitioned Parquet into a temporary folder first:
    # {output_base_dir}/{region}/_tmp_v{version}/year=YYYY/
    tmp_partition_dir.mkdir(parents=True, exist_ok=True)
    df.write_parquet(
        tmp_partition_dir,
        partition_by=["year"],
        compression="zstd"
    )

    # Integrity verification 1: per-year row counts, source vs partitioned
    written_rows_per_year = partitioned_rows_per_year(tmp_partition_dir)
    if written_rows_per_year != source_rows_per_year:
        mismatched = sorted(
            y for y in set(source_rows_per_year) | set(written_rows_per_year)
            if source_rows_per_year.get(y) != written_rows_per_year.get(y)
        )
        raise ValueError(
            f"Per-year row count mismatch for {region} v{file_version} in years {mismatched}. "
            f"Source file and {tmp_partition_dir} retained for debugging."
        )

    # Integrity verification 2: column names and types, source vs partitioned
    written_schema = partitioned_schema(tmp_partition_dir)
    if written_schema != source_schema:
        raise ValueError(
            f"Column/type mismatch for {region} v{file_version}.\n"
            f"  Source:      {source_schema}\n  Partitioned: {written_schema}\n"
            f"Source file and {tmp_partition_dir} retained for debugging."
        )

    # Integrity verification 3: per-year row counts against the GEE asset
    if VERIFY_AGAINST_GEE:
        gee_counts = gee_rows_per_year(region, file_version)
        if gee_counts != source_rows_per_year:
            mismatched = sorted(
                y for y in set(gee_counts) | set(source_rows_per_year)
                if gee_counts.get(y) != source_rows_per_year.get(y)
            )
            raise ValueError(
                f"Download does not match GEE asset for {region} v{file_version} in years {mismatched}. "
                f"Source file and {tmp_partition_dir} retained for debugging."
            )

    print(f"  Verification passed ({raw_row_count:,} rows, {len(source_rows_per_year)} years"
          f"{', matches GEE' if VERIFY_AGAINST_GEE else ''}).")

    # All checks passed: move the complete partition into its final place in one rename
    tmp_partition_dir.rename(region_partition_dir)
    print(f"  Successfully partitioned '{region}' v{file_version} into: {region_partition_dir}/")

    # 1. Delete the primary processed source file
    print(f"  Deleting processed source file: {filepath.name}...")
    filepath.unlink()

    # 2. Check and delete counterpart extension if present (e.g., .csv alongside .parquet)
    counterpart_ext = '.csv' if filepath.suffix.lower() == '.parquet' else '.parquet'
    counterpart_file = filepath.with_suffix(counterpart_ext)
    if counterpart_file.exists():
        print(f"  Deleting corresponding counterpart: {counterpart_file.name}...")
        counterpart_file.unlink()

    print(f"  Freed disk space successfully for '{region}'.")

Discovered 0 source file(s): []


5. Remove older versions
Deletes `v<N>` folders other than the version in `samples_version_map`, but only for regions whose current version is present locally. Runs as a dry run first: check the list, then set `DRY_RUN_CLEANUP = False`.

In [11]:
DRY_RUN_CLEANUP = True  # set to False after checking the planned deletions

for region, version in samples_version_map.items():
    region_dir = DATA_DIR / region
    if not region_dir.exists():
        continue

    current_dir = region_data_path(region, version)
    old_dirs = sorted(
        d for d in region_dir.iterdir()
        if d.is_dir() and re.fullmatch(r"v\d+", d.name) and d != current_dir
    )
    if not old_dirs:
        continue

    if not any(current_dir.glob("year=*")):
        print(f"'{region}': keeping {[d.name for d in old_dirs]}, current v{version} is not downloaded yet.")
        continue

    print(f"'{region}': delete {[d.name for d in old_dirs]} (keeping v{version})")
    if not DRY_RUN_CLEANUP:
        for d in old_dirs:
            shutil.rmtree(d)

print("\nDry run only, nothing deleted." if DRY_RUN_CLEANUP else "\nCleanup done.")


Dry run only, nothing deleted.
